In [98]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# for one hot encoding 
from sklearn.preprocessing import OneHotEncoder 

In [99]:
df = pd.read_csv("../data/preprocessed/supply_chain_disruption.csv")

initial views at the data

In [100]:
df.shape

(31300, 48)

In [101]:
df.head(10)

,date,route_id,route_status,origin_country,destination_country,shipping_method,trade_route_type,distance_km,estimated_transit_days,shipping_delay_days,...,origin_iso3,destination_region,destination_event_region,destination_income_group,destination_port_capacity_index,destination_logistics_performance_index,destination_trade_dependency_score,destination_gdp_per_capita,destination_population,destination_iso3
0,01/04/2015,R1,Delayed,India,United States,Rail,Consumer Goods,8746,19,7.55,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA
1,01/11/2015,R1,Delayed,India,United States,Rail,Consumer Goods,8746,19,9.33,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA
2,01/18/2015,R1,Normal,India,United States,Rail,Consumer Goods,8746,19,3.67,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA
3,01/25/2015,R1,Delayed,India,United States,Rail,Consumer Goods,8746,19,9.22,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA
4,02/01/2015,R1,Delayed,India,United States,Rail,Consumer Goods,8746,19,6.36,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA
5,02/08/2015,R1,Delayed,India,United States,Rail,Consumer Goods,8746,19,10.61,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA
6,02/15/2015,R1,Delayed,India,United States,Rail,Consumer Goods,8746,19,7.56,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA
7,02/22/2015,R1,Delayed,India,United States,Rail,Consumer Goods,8746,19,8.22,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA
8,03/01/2015,R1,Delayed,India,United States,Rail,Consumer Goods,8746,19,8.00,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA
9,03/08/2015,R1,Normal,India,United States,Rail,Consumer Goods,8746,19,3.08,...,IND,North America,North America,Low income,21.06,33.87,97.9,63955,250467210,USA


In [102]:
df.columns

Index(['date', 'route_id', 'route_status', 'origin_country',
       'destination_country', 'shipping_method', 'trade_route_type',
       'distance_km', 'estimated_transit_days', 'shipping_delay_days',
       'port_congestion_index', 'container_availability_index',
       'weather_disruption_score', 'geopolitical_risk_score',
       'trade_volume_tonnes', 'freight_cost_usd', 'carbon_emissions_tonnes',
       'origin_active_event_count', 'origin_active_event_risk',
       'destination_active_event_count', 'destination_active_event_risk',
       'covid_supply_shock_active', 'russia_ukraine_war_active',
       'events_warmup', 'fuel_cost_index', 'commodity_price_index',
       'natural_gas_price', 'steel_price', 'wheat_price', 'copper_price',
       'origin_region', 'origin_event_region', 'origin_income_group',
       'origin_port_capacity_index', 'origin_logistics_performance_index',
       'origin_trade_dependency_score', 'origin_gdp_per_capita',
       'origin_population', 'origin_iso3'

Dropping rows with the Disrupted routed status, as it only accounts for 3% of all total data

In [103]:
# use the inplace=True
# From CA Ashok:
# do a model with and without
# predict disruptions, not on time or not
# and return biggest indicators of such
# distribution center at A, weather disruption at A, asks about A, model says no, weather thing, go to B

# indices_to_drop = df[df['route_status'] == "Disrupted"].index
# df.drop(indices_to_drop, inplace=True)


In [104]:
df.shape

(31300, 48)

In [105]:
df.iloc[30000:] 
# checking that data is ordered chronologically

,date,route_id,route_status,origin_country,destination_country,shipping_method,trade_route_type,distance_km,estimated_transit_days,shipping_delay_days,...,origin_iso3,destination_region,destination_event_region,destination_income_group,destination_port_capacity_index,destination_logistics_performance_index,destination_trade_dependency_score,destination_gdp_per_capita,destination_population,destination_iso3
30000,02/01/2026,R48,Delayed,Brazil,Japan,Rail,Energy,1103,22,5.3600,...,BRA,East Asia & Pacific,Asia,Upper middle income,65.06,34.58,27.25,68969,435285667,JPN
30001,02/08/2026,R48,Delayed,Brazil,Japan,Rail,Energy,1103,22,8.3500,...,BRA,East Asia & Pacific,Asia,Upper middle income,65.06,34.58,27.25,68969,435285667,JPN
30002,02/15/2026,R48,Normal,Brazil,Japan,Rail,Energy,1103,22,0.0000,...,BRA,East Asia & Pacific,Asia,Upper middle income,65.06,34.58,27.25,68969,435285667,JPN
30003,02/22/2026,R48,Normal,Brazil,Japan,Rail,Energy,1103,22,3.6200,...,BRA,East Asia & Pacific,Asia,Upper middle income,65.06,34.58,27.25,68969,435285667,JPN
30004,03/01/2026,R48,Delayed,Brazil,Japan,Rail,Energy,1103,22,8.9200,...,BRA,East Asia & Pacific,Asia,Upper middle income,65.06,34.58,27.25,68969,435285667,JPN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31295,11/29/2026,R50,Delayed,Brazil,United States,Sea,Consumer Goods,8070,28,9.2475,...,BRA,North America,North America,Low income,21.06,33.87,97.90,63955,250467210,USA
31296,12/06/2026,R50,Delayed,Brazil,United States,Sea,Consumer Goods,8070,28,11.7180,...,BRA,North America,North America,Low income,21.06,33.87,97.90,63955,250467210,USA
31297,12/13/2026,R50,Disrupted,Brazil,United States,Sea,Consumer Goods,8070,28,12.6495,...,BRA,North America,North America,Low income,21.06,33.87,97.90,63955,250467210,USA
31298,12/20/2026,R50,Delayed,Brazil,United States,Sea,Consumer Goods,8070,28,11.1240,...,BRA,North America,North America,Low income,21.06,33.87,97.90,63955,250467210,USA


Count number of unique values, will need to one hot encode things

In [106]:
df.nunique()

date                                         626
route_id                                      50
route_status                                   3
origin_country                                10
destination_country                           10
shipping_method                                4
trade_route_type                               5
distance_km                                   50
estimated_transit_days                        31
shipping_delay_days                         2235
port_congestion_index                       9594
container_availability_index                4994
weather_disruption_score                    9585
geopolitical_risk_score                     9559
trade_volume_tonnes                        31145
freight_cost_usd                           30361
carbon_emissions_tonnes                    30928
origin_active_event_count                     93
origin_active_event_risk                    2455
destination_active_event_count                93
destination_active_e

In [107]:
def print_word_count():
        wordy_cols = list(df.select_dtypes(include=["object"]).columns)
        print(wordy_cols)
        for w in wordy_cols:
                print(f"{w}: {df[w].nunique()}")

In [108]:
print_word_count()

['date', 'route_id', 'route_status', 'origin_country', 'destination_country', 'shipping_method', 'trade_route_type', 'origin_region', 'origin_event_region', 'origin_income_group', 'origin_iso3', 'destination_region', 'destination_event_region', 'destination_income_group', 'destination_iso3']
date: 626
route_id: 50
route_status: 3
origin_country: 10
destination_country: 10
shipping_method: 4
trade_route_type: 5
origin_region: 5
origin_event_region: 4
origin_income_group: 4
origin_iso3: 10
destination_region: 5
destination_event_region: 4
destination_income_group: 4
destination_iso3: 10


get rid of: origin iso3, route id, destination iso3, desitnation region (cause its grouped and similar to destination event region)
discussed on call

In [109]:
df.drop(["route_id", "origin_iso3", "destination_iso3", "destination_region"], axis=1, inplace=True)

In [110]:
df.shape

(31300, 44)

In [111]:
df.head(10)

,date,route_status,origin_country,destination_country,shipping_method,trade_route_type,distance_km,estimated_transit_days,shipping_delay_days,port_congestion_index,...,origin_trade_dependency_score,origin_gdp_per_capita,origin_population,destination_event_region,destination_income_group,destination_port_capacity_index,destination_logistics_performance_index,destination_trade_dependency_score,destination_gdp_per_capita,destination_population
0,01/04/2015,Delayed,India,United States,Rail,Consumer Goods,8746,19,7.55,82.22,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210
1,01/11/2015,Delayed,India,United States,Rail,Consumer Goods,8746,19,9.33,72.01,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210
2,01/18/2015,Normal,India,United States,Rail,Consumer Goods,8746,19,3.67,44.64,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210
3,01/25/2015,Delayed,India,United States,Rail,Consumer Goods,8746,19,9.22,97.82,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210
4,02/01/2015,Delayed,India,United States,Rail,Consumer Goods,8746,19,6.36,73.21,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210
5,02/08/2015,Delayed,India,United States,Rail,Consumer Goods,8746,19,10.61,74.91,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210
6,02/15/2015,Delayed,India,United States,Rail,Consumer Goods,8746,19,7.56,80.74,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210
7,02/22/2015,Delayed,India,United States,Rail,Consumer Goods,8746,19,8.22,40.05,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210
8,03/01/2015,Delayed,India,United States,Rail,Consumer Goods,8746,19,8.00,56.59,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210
9,03/08/2015,Normal,India,United States,Rail,Consumer Goods,8746,19,3.08,51.89,...,23.73,60735,430389014,North America,Low income,21.06,33.87,97.9,63955,250467210


In [112]:
print_word_count()

['date', 'route_status', 'origin_country', 'destination_country', 'shipping_method', 'trade_route_type', 'origin_region', 'origin_event_region', 'origin_income_group', 'destination_event_region', 'destination_income_group']
date: 626
route_status: 3
origin_country: 10
destination_country: 10
shipping_method: 4
trade_route_type: 5
origin_region: 5
origin_event_region: 4
origin_income_group: 4
destination_event_region: 4
destination_income_group: 4


Time to one hote encode things now

In [114]:
wordy_cols = list(df.select_dtypes(include=["object"]).columns)
wordy_cols.remove('date')

df_enc = pd.get_dummies(df, columns=wordy_cols,dtype='int')
df_enc.head()

,date,distance_km,estimated_transit_days,shipping_delay_days,port_congestion_index,container_availability_index,weather_disruption_score,geopolitical_risk_score,trade_volume_tonnes,freight_cost_usd,...,origin_income_group_Lower middle income,origin_income_group_Upper middle income,destination_event_region_Asia,destination_event_region_Europe,destination_event_region_North America,destination_event_region_South America,destination_income_group_High income,destination_income_group_Low income,destination_income_group_Lower middle income,destination_income_group_Upper middle income
0,01/04/2015,8746,19,7.55,82.22,71.42,59.51,57.15,8418.69,4586.66,...,0,0,0,0,1,0,0,1,0,0
1,01/11/2015,8746,19,9.33,72.01,79.27,91.27,41.47,9343.00,4574.70,...,0,0,0,0,1,0,0,1,0,0
2,01/18/2015,8746,19,3.67,44.64,60.16,46.69,79.69,7090.69,4520.46,...,0,0,0,0,1,0,0,1,0,0
3,01/25/2015,8746,19,9.22,97.82,66.43,56.39,67.15,7829.79,4696.95,...,0,0,0,0,1,0,0,1,0,0
4,02/01/2015,8746,19,6.36,73.21,96.20,95.92,7.59,11339.59,4507.98,...,0,0,0,0,1,0,0,1,0,0


In [115]:
df_enc.columns

Index(['date', 'distance_km', 'estimated_transit_days', 'shipping_delay_days',
       'port_congestion_index', 'container_availability_index',
       'weather_disruption_score', 'geopolitical_risk_score',
       'trade_volume_tonnes', 'freight_cost_usd', 'carbon_emissions_tonnes',
       'origin_active_event_count', 'origin_active_event_risk',
       'destination_active_event_count', 'destination_active_event_risk',
       'covid_supply_shock_active', 'russia_ukraine_war_active',
       'events_warmup', 'fuel_cost_index', 'commodity_price_index',
       'natural_gas_price', 'steel_price', 'wheat_price', 'copper_price',
       'origin_port_capacity_index', 'origin_logistics_performance_index',
       'origin_trade_dependency_score', 'origin_gdp_per_capita',
       'origin_population', 'destination_port_capacity_index',
       'destination_logistics_performance_index',
       'destination_trade_dependency_score', 'destination_gdp_per_capita',
       'destination_population', 'route_statu

In [ ]:
# df[df_enc["origin_country_United Kingdom"] == 1].count()
# random test 

DO CORRELATION!!! (but first one hot encode route_status)

note todo, after cleaning, separate data into train and text based on dates.